## Partition

Divide our big work into small works.

PySpark doesn't keep a giant file. Instead, it breaks the data down into smaller pieces (partitions) so that multiple computers (worker nodes) can work on different pieces of the data at the exact same time.

## Transformation
It does not calculate anything. It just build a blueprint or a plan of what you want to do. 

If a method modifies the structure of the data (adding columns, filtering rows, grouping things), it is a transformation.

It means 
* prepare the work
* modify data logic
* no actual exexcution yet

* **Modifying Columns**: .withColumn(), .drop(), .select()
* **Changing Rows**: .filter(), .where(), .distinct(), .dropDuplicates()
* **Combining Data**: .join(), .union()
* **Organizing Data**: .groupBy(), .orderBy(), .sort()

**Rule of Thumb**: If the method outputs a new DataFrame, it is always a Transformation

### Two types
1. Narrow Transformation
1. Wide Transformation

* Narrow Transformation:- one to one(fast)

    Here spark doesn't move data between partitions. Each partition works independently. It means when spark divides our data into partitions, one partition data has one result, it doesn't communicate with other paertitions.

        * Eg. filter(), select(), withColumn(), map(), flatMap()

* Wide Transformation:- one to N(slow)

    Here spark must move data between partitions(called shuffling). If we apply groupby,  partitions communicate with each other, collect same groups together and move data across partitions.
    
        * Eg. groupBy(), join(), distinct(), orderBy(), repartition(), dropDuplicates()

In [0]:
df = spark.read.table('samples.bakehouse.sales_customers')
display(df.limit(5)) # action

In [0]:
# Narrow Transformations
from pyspark.sql import functions as F

female_df = df.filter(df['gender'] == 'Female')
name_df = df.select('first_name', 'last_name') 

new_df = df.withColumn('full name', F.concat(F.col('first_name'),  F.lit(' '), F.col('last_name')))

In [0]:
# wide transformations 
city_df = df.groupBy("city").agg(F.count('*').alias('tot_emp_in_each_city'))

## Action
To trigger the execution we need is called action. It says, execute the instructions right now, and give me the result.

* **Viewing/Printing data**: .show(), display(), .printSchema()
* **Counting or Math**: .count(), .sum() (when used directly on a DataFrame, not inside a group)
* **Bringing data to the driver**: .collect(), .take(), .first(), .head()
* **Saving data**: .write, .save()

In [0]:
# take(n):- get the 'first' n rows of a DataFrame and return them as a list
df.take(3)

In [0]:
# extract values from the very first record
display(df.first())

In [0]:
display(df.tail(2))

In [0]:
display(female_df.limit(5))
display(name_df.limit(5))
display(new_df.limit(5))
display(city_df.limit(5))

In [0]:
# collect- Brings ALL data to driver/main machine
## Note: If data is huge, don't use it because all data comes to one machine cause slow performance and memory issue.
df.collect()[:3]

In [0]:
[row['first_name'] for row in df.collect()[:3]]

In [0]:
# display(df.select("city").distinct().rdd.flatMap(lambda x: x).collect()) # rdd doesn't work on 'serverless'
unique_vals = [
    row["city"]
    for row in df.select("city").distinct().collect()
]

print(unique_vals[:30])

### Spark prefers Lazy Evaluation
Spark waits until an action is called. This allows Spark to optimize, plan and use the resources properly.